<a href="https://colab.research.google.com/github/gocenalper/BigramModels/blob/main/make_more.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [ ]:
import wandb
wandb.login()

In [ ]:
from pathlib import Path
import urllib.request

In [ ]:
_BASE_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()
DATA_PATH = _BASE_DIR / "data" / "names.txt"
DATA_URL = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"

# Create the data directory if it doesn't exist
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

# Download the file if it doesn't exist locally
if not DATA_PATH.exists():
    print(f"Downloading {DATA_URL} to {DATA_PATH}...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Download complete.")

In [ ]:
words = DATA_PATH.read_text(encoding="utf-8").splitlines()

words[:10]

In [ ]:
len(words)

In [ ]:
b = {}
for w in words:
  chs = ['<S>'] + list(w) + ['<E>']
  for ch1, ch2 in zip(chs, chs[1:]):
    bigram = (ch1, ch2)
    b[bigram] = b.get(bigram, 0) + 1

In [ ]:
chars = sorted(b.items(), key = lambda kv: -kv[1])

In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)

In [ ]:
chars = sorted(list(set(''.join(words))))

stoi = {ch: i+1 for i, ch in enumerate(chars)}
stoi['.'] = 0

itos = {i: ch for ch, i in stoi.items()}

In [ ]:
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline
plt.figure(figsize=(16, 16))
plt.imshow(N, cmap='Blues')

for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off')

In [ ]:
p = N[0].float()
p = p/p.sum()
p

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(10):
  out = []
  ix = 0
  while True:
    p = N[ix].float()
    p = p/p.sum()
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))

### Olasılık Matrisini (P) Broadcasting ile Oluşturmak
Burada PyTorch'un **broadcasting** kurallarını kullanıyoruz. `P.sum(1, keepdim=True)` ifadesi bize `27x1` boyutunda bir vektör döndürür. `P` ise `27x27` boyutundadır. PyTorch boyutları sağdan sola doğru eşleştirir, böylece `27x1`'lik bu matris, bölme işlemi sırasında otomatik olarak kopyalanarak (broadcast) `27x27`'lik tüm matrise uygulanır.

Ayrıca ileride `0` olan olasılıkların (ve bunların logaritmasının eksi sonsuz olmasının) önüne geçmek için tüm sayılara `+1` (Laplace Smoothing) ekliyoruz.

In [ ]:
# 1 ekleyerek model smoothing (yumuşatma) yapıyoruz
P = (N+1).float()

# Broadcasting sayesinde tüm matrisi tek seferde normalize ediyoruz:
# P = 27x27
# P.sum(1, keepdim=True) = 27x1
P /= P.sum(1, keepdim=True)

print(f"P shape: {P.shape}")

Oluşturduğumuz `P` matrisi ile örneklem (sampling) döngümüzü güncelliyoruz. Artık döngü içinde her seferinde `.sum()` hesaplamamıza gerek kalmadı, kodu çok daha performanslı hale getirdik:

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(10):
  out = []
  ix = 0
  while True:
    # Artık direkt P matrisinden ilgili satırı çekiyoruz
    p = P[ix]

    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))

### Modelin Başarısını Ölçmek: Negatif Log-Olabilirlik (NLL)

Artık modelimizden kelimeler üretebiliyoruz. Peki bu model **ne kadar iyi**?
Bunu ölçmek için veri setimizdeki kelimeleri alıp, modelimizin bu kelimelerdeki harf geçişlerine (bigram'lara) ne kadar olasılık verdiğine bakarız.

İyi bir model, veri setindeki gerçek geçişlere yüksek olasılık vermelidir.
- Olasılıkların çarpımı **Likelihood** (Olabilirlik) olarak adlandırılır.
- Olasılıklar 0 ile 1 arasında olduğu için, bunları çarptıkça sayı çok küçülür. Bu yüzden **Log-Likelihood** (Olasılıkların logaritmalarının toplamı) kullanırız.
- Makine öğrenmesinde kayıp (loss) değerini **küçültmek** istediğimiz için, bu değeri eksi ile çarparak **Negatif Log-Olabilirlik (Negative Log Likelihood - NLL)** elde ederiz. Ortalama NLL ne kadar düşükse, modelimiz o kadar iyidir!

In [ ]:
# Örnek olarak ilk 3 kelimenin olasılıklarına ve log-olasılıklarına bakalım:
log_likelihood = 0.0
n = 0

for w in words[:3]:
  print(f"--- {w} ---")
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1
    print(f'{ch1}{ch2}: olasılık={prob:.4f}, log-olasılık={logprob:.4f}')

print('='*20)
print(f'log_likelihood = {log_likelihood:.4f}')
nll = -log_likelihood
print(f'negatif log_likelihood = {nll:.4f}')
print(f'Ortalama Kayıp (Loss / NLL) = {nll/n:.4f}')

Şimdi bu işlemi **tüm veri setimiz (words)** için yapalım ve modelimizin genel performansını (loss değerini) görelim.

Karpathy'nin makemore serisinde temel loss fonksiyonumuz budur. Tüm eğitim boyunca amacımız bu ortalama NLL değerini (yaklaşık 2.45 civarı çıkacaktır) daha gelişmiş yapay sinir ağı modelleri kurarak aşağı çekmek olacak.

In [ ]:
# Tüm veri seti için NLL (Loss) hesaplama
log_likelihood = 0.0
n = 0

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1

nll = -log_likelihood
print(f'Tüm veri seti için Ortalama Kayıp (Loss/NLL) = {nll/n:.4f}')

### Model Smoothing (Yumuşatma) Neden Gerekliydi?

Videodaki sıraya göre, NLL hesapladıktan sonra modelimizi hiç görmediği bir harf dizilimiyle test edersek ne olacağına bakmamız gerekiyor. Mesela **"andrejq"** kelimesi.

Veri setimizde 'j' harfinden sonra 'q' hiç gelmiyor. Eğer biz matrisi oluştururken `+1` eklememiş olsaydık, bu geçişin olasılığı `0` olacaktı ve logaritması `-sonsuz` çıkacağı için modelin kaybı (loss) patlayacaktı.

Biz daha önce `P = (N+1).float()` diyerek (buna **Laplace Smoothing** denir) her geçişe en az 1 sayı eklediğimiz için bu sonsuz kayıp hatasından kurtulmuş olduk. Görelim:

In [ ]:
# "andrejq" kelimesini test edelim
log_likelihood = 0.0
n = 0
for w in ["andrejq"]:
  print(f"--- {w} ---")
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1
    print(f'{ch1}{ch2}: olasılık={prob:.4f}, log-olasılık={logprob:.4f}')

nll = -log_likelihood
print(f'Ortalama Kayıp = {nll/n:.4f}')
print("\n'jq' geçişine dikkat! Olasılığı 0 değil, +1 smoothing sayesinde çok küçük de olsa bir olasılığı var ve loss sonsuz çıkmıyor.")

### Bölüm 2: Yapay Sinir Ağı (Neural Network) Yaklaşımı

İstatistiksel (sayma tabanlı) Bigram modelimizi bitirdik ve smoothing ile pürüzsüzleştirdik. Karpathy'nin videosunda artık yepyeni bir sayfaya geçiyoruz: Aynı problemi **Yapay Sinir Ağı** ile çözeceğiz!

Bunun için öncelikle ağımızı eğiteceğimiz veri setimizi hazırlamalıyız. Girdilerimiz (`x`) ilk harf, hedeflerimiz (`y`) ise ondan sonra gelmesi gereken ikinci harf olacak.

In [ ]:
# Sinir ağı için eğitim setini (training set) oluşturalım
xs, ys = [], []

for w in words[:1]: # Şimdilik sadece ilk kelime ("emma") üzerinden görelim
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    print(f"{ch1} ---> {ch2}")
    xs.append(ix1)
    ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

print("\nGirdiler (xs):", xs)
print("Hedefler (ys):", ys)

### One-Hot Encoding
Sinir ağına vereceğimiz girdilerin (harflerin) aralarındaki anlamsız büyüklük-küçüklük ilişkisini kaldırmak için onları 27 boyutlu (0 ve 1'lerden oluşan) vektörlere çeviriyoruz.

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

# xs tensorünü one-hot vektörlere çevirme
# 27 farklı karakterimiz olduğu için num_classes=27
xenc = F.one_hot(xs, num_classes=27).float()

print("xenc boyutu (shape):", xenc.shape)
print("xenc veri tipi:", xenc.dtype)

# Görselleştirme (Sarı noktalar 1'leri, mor kısımlar 0'ları temsil eder)
plt.imshow(xenc)

### Sinir Ağının İlk Katmanı: Ağırlıklar (Weights) ve Logits

Rastgele sayılardan oluşan 27x27'lik bir ağırlık matrisi (`W`) oluşturuyoruz. Bu bizim en temel (tek katmanlı) sinir ağımız olacak.

Girdilerimizle (`xenc`) ağırlıkları çarptığımızda (`@` operatörü), her bir giriş harfi için 27 farklı nöronun ürettiği çıktıları elde edeceğiz.

In [ ]:
# Ağırlık matrisini rastgele sayılarla başlatalım (Normal dağılım)
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g)

# İleri yayılım (Forward pass): Girdiler x Ağırlıklar
# (5, 27) @ (27, 27) = (5, 27)
logits = xenc @ W

print("Ağırlık matrisi (W) boyutu:", W.shape)
print("Çıktı (logits) matrisi boyutu:", logits.shape)
print("\nİlk örneğimizin (ilk harfin) logits değerleri:\n", logits[0])

### Logits'ten Olasılıklara (Softmax)

Çok iyi bir noktaya değindiniz! Ancak ufak ama çok kritik bir detay var: `xenc @ W` işleminden çıkan sonuçlar şu an **olasılık değil**, sadece modelin ürettiği **ham skorlar (logits)**. Bazıları negatif, bazıları pozitif.

Girdilerimiz (`xenc`) One-Hot yapısında (sadece bir elemanı 1, gerisi 0) olduğu için, `xenc @ W` işlemi aslında W matrisindeki ilgili harfin satırını **olduğu gibi çekip almaktan** (bir nevi filtrelemekten) ibarettir.

Modelin bunu öğrenebilmesi için:
1. Bu ham skorları (logits) **olasılıklara (0 ile 1 arasına)** dönüştürmemiz gerekir.
2. Elde ettiğimiz olasılıkları, gerçek harfle karşılaştırıp hatamızı bulacağız.
3. Geriye doğru gidip **W matrisindeki ağırlıkları güncelleyeceğiz** (Backpropagation).

Şimdi bu ham skorları olasılıklara dönüştürelim. Bu işleme sinir ağlarında **Softmax** adı verilir:

In [ ]:
# 1. Adım: Skorları pozitife çevirmek için e tabanında üslerini alıyoruz (exp)
# Bu adım aslında istatistiksel modeldeki 'N' (sayma/counts) matrisine benzer bir etki yaratır.
counts = logits.exp()

# 2. Adım: Her satırı kendi toplamına bölerek olasılık haline getiriyoruz (0-1 arası ve toplamı 1)
# Bu adım istatistiksel modeldeki 'P' matrisini (normalizasyon) oluşturmaya benzer.
probs = counts / counts.sum(1, keepdims=True)

print("Olasılık matrisinin boyutu (probs):", probs.shape)
print("\nİlk örneğimizin olasılık değerleri:\n", probs[0])
print("\nBu olasılıkların toplamı (1.0 olmalı):", probs[0].sum().item())

### Kayıp (Loss) Değerini Hesaplamak

Artık elimizde her bir giriş harfi için 27 farklı harfin gelme olasılığı var. Peki bizim modelimiz doğru harflere ne kadar olasılık vermiş?

Doğru harflerimiz (hedefler) `ys` tensoründe duruyor: `[5, 13, 13, 1, 0]`
- 0. girdi ('.') için 5. harf ('e') gelmeli.
- 1. girdi ('e') için 13. harf ('m') gelmeli.

`probs` matrisinden bu doğru harflere karşılık gelen olasılıkları çekip (plucking), Negative Log-Likelihood (NLL) hesaplayalım.

In [ ]:
# 5 örneğimiz var
num_examples = 5

# PyTorch'un indeksleme yeteneğini kullanarak doğru harflerin olasılıklarını çekiyoruz
# probs[0, 5], probs[1, 13], probs[2, 13] ... gibi
correct_probs = probs[torch.arange(num_examples), ys]
print("Doğru harflere verilen olasılıklar:", correct_probs)

# Logaritmalarını alıyoruz
log_probs = torch.log(correct_probs)

# Negatif ortalamasını (NLL) alarak Loss değerini buluyoruz
loss = -log_probs.mean()
print(f"\nLoss (Kayıp) değerimiz: {loss.item():.4f}")

<!-- Silindi -->